In [1]:
from transformers import BertTokenizer, BertModel
import pandas as pd
import numpy as np
import nltk
import torch

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model = BertModel.from_pretrained('bert-base-uncased',
                                  output_hidden_states = True,
                                  )



In [5]:

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
# def bert_text_preparation(text, tokenizer):
#     """Preparing the input for BERT
    
#     Takes a string argument and performs
#     pre-processing like adding special tokens,
#     tokenization, tokens to ids, and tokens to
#     segment ids. All tokens are mapped to seg-
#     ment id = 1.
    
#     Args:
#         text (str): Text to be converted
#         tokenizer (obj): Tokenizer object
#             to convert text into BERT-re-
#             adable tokens and ids
        
#     Returns:
#         list: List of BERT-readable tokens
#         obj: Torch tensor with token ids
#         obj: Torch tensor segment ids
    
    
#     """
#     marked_text = "[CLS] " + text + " [SEP]"
#     tokenized_text = tokenizer.tokenize(marked_text)
#     indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
#     segments_ids = [1]*len(indexed_tokens)

#     # Convert inputs to PyTorch tensors
#     tokens_tensor = torch.tensor([indexed_tokens])
#     segments_tensors = torch.tensor([segments_ids])

#     return tokenized_text, tokens_tensor, segments_tensors

def bert_text_preparation(text, tokenizer, max_length=512):
    encoded_input = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=max_length,
        truncation=True,
        return_tensors="pt"
    )
    tokens_tensor = encoded_input["input_ids"]
    segments_tensors = encoded_input["token_type_ids"]

    tokenized_text = tokenizer.convert_ids_to_tokens(tokens_tensor[0])
    return tokenized_text, tokens_tensor, segments_tensors


In [7]:

    
def get_bert_embeddings(tokens_tensor, segments_tensors, model):
    """Get embeddings from an embedding model
    
    Args:
        tokens_tensor (obj): Torch tensor size [n_tokens]
            with token ids for each token in text
        segments_tensors (obj): Torch tensor size [n_tokens]
            with segment ids for each token in text
        model (obj): Embedding model to generate embeddings
            from token and segment ids
    
    Returns:
        list: List of list of floats of size
            [n_tokens, n_embedding_dimensions]
            containing embeddings for each token
    
    """
    
    # Gradient calculation id disabled
    # Model is in inference mode
    with torch.no_grad():
        outputs = model(tokens_tensor, segments_tensors)
        # Removing the first hidden state
        # The first state is the input state
        hidden_states = outputs[2][1:]

    # Getting embeddings from the final BERT layer
    token_embeddings = hidden_states[-1]
    # Collapsing the tensor into 1-dimension
    token_embeddings = torch.squeeze(token_embeddings, dim=0)
    # Converting torchtensors to lists
    list_token_embeddings = [token_embed.tolist() for token_embed in token_embeddings]

    return list_token_embeddings




In [8]:
import pandas

In [2]:
df = pd.read_csv("dataset/combined.csv")

In [28]:
texts = df["Abstract"][:50]

In [29]:
abstract_word_embedding = []

In [30]:
texts[49]

'Deep Learning can significantly benefit cancer proteomics and genomics. In\nthis study, we attempt to determine a set of critical proteins that are\nassociated with the FLT3-ITD mutation in newly-diagnosed acute myeloid leukemia\npatients. A Deep Learning network consisting of autoencoders forming a\nhierarchical model from which high-level features are extracted without labeled\ntraining data. Dimensional reduction reduced the number of critical proteins\nfrom 231 to 20. Deep Learning found an excellent correlation between FLT3-ITD\nmutation with the levels of these 20 critical proteins (accuracy 97%,\nsensitivity 90%, specificity 100%). Our Deep Learning network could hone in on\n20 proteins with the strongest association with FLT3-ITD. The results of this\nstudy allow a novel approach to determine critical protein pathways in the\nFLT3-ITD mutation, and provide proof-of-concept for an accurate approach to\nmodel big data in cancer proteomics and genomics.'

In [32]:
for text in texts:
    tokenized_text, tokens_tensor, segments_tensors = bert_text_preparation(text, tokenizer)
    list_token_embeddings = get_bert_embeddings(tokens_tensor, segments_tensors, model)

    abstract_word_embedding.append(list_token_embeddings)


In [36]:
len(tokens_tensor)

1

In [44]:
len(abstract_word_embedding)

50

In [54]:
arr = np.array(abstract_word_embedding[2])
arr.shape

(208, 768)

(188, 768)

In [46]:
# arr = np.array(abstract_word_embedding)
np.save('abstract_word.npy', arr) 

In [55]:
p = np.load("abstract_word.npy")

In [56]:
p 


array([[-0.17365143,  0.31911486, -0.18623812, ..., -0.37519705,
         0.10291668, -0.16890082],
       [-0.24774763,  0.32880315, -0.35583481, ..., -0.42945603,
         0.07558236, -0.1868183 ],
       [-0.47712842,  0.49237248, -0.42837426, ..., -0.47571108,
         0.08668672, -0.08852481],
       ...,
       [-0.40491742,  0.5326826 , -0.40256512, ..., -0.4929361 ,
         0.01889027, -0.36566666],
       [-0.41039702,  0.25452375, -0.19722369, ..., -0.18292914,
         0.07959157, -0.47037399],
       [-0.55124962,  0.42968804, -0.24920055, ..., -0.4305056 ,
         0.0335878 , -0.4335441 ]])

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [4]:
print(' Original: ', df["Abstract"].values[0])

# Print the sentence split into tokens.

 Original:  Data science is the business of learning from data, which is traditionally
the business of statistics. Data science, however, is often understood as a
broader, task-driven and computationally-oriented version of statistics. Both
the term data science and the broader idea it conveys have origins in
statistics and are a reaction to a narrower view of data analysis. Expanding
upon the views of a number of statisticians, this paper encourages a big-tent
view of data analysis. We examine how evolving approaches to modern data
analysis relate to the existing discipline of statistics (e.g. exploratory
analysis, machine learning, reproducibility, computation, communication and the
role of theory). Finally, we discuss what these trends mean for the future of
statistics by highlighting promising directions for communication, education
and research.


Tokenized:  ['data', 'science', 'is', 'the', 'future', 'of', 'ai']
Token IDs:  [2951, 2671, 2003, 1996, 2925, 1997, 9932]


In [14]:
text = "I would like to choose my profession as a datascientist"
print('Tokenized: ', tokenizer.tokenize(text))

# Print the sentence mapped to token ids.
print('Token IDs: ', tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

Tokenized:  ['i', 'would', 'like', 'to', 'choose', 'my', 'profession', 'as', 'a', 'data', '##sc', '##ient', '##ist']
Token IDs:  [1045, 2052, 2066, 2000, 5454, 2026, 9518, 2004, 1037, 2951, 11020, 11638, 2923]


In [25]:
text = df["Abstract"].values[0]
text

'Data science is the business of learning from data, which is traditionally\nthe business of statistics. Data science, however, is often understood as a\nbroader, task-driven and computationally-oriented version of statistics. Both\nthe term data science and the broader idea it conveys have origins in\nstatistics and are a reaction to a narrower view of data analysis. Expanding\nupon the views of a number of statisticians, this paper encourages a big-tent\nview of data analysis. We examine how evolving approaches to modern data\nanalysis relate to the existing discipline of statistics (e.g. exploratory\nanalysis, machine learning, reproducibility, computation, communication and the\nrole of theory). Finally, we discuss what these trends mean for the future of\nstatistics by highlighting promising directions for communication, education\nand research.'

In [24]:
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print(tokenizer.tokenize(text))
print(
tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

['data', 'science', 'is', 'the', 'business', 'of', 'learning', 'from', 'data', ',', 'which', 'is', 'traditionally', 'the', 'business', 'of', 'statistics', '.', 'data', 'science', ',', 'however', ',', 'is', 'often', 'understood', 'as', 'a', 'broader', ',', 'task', '-', 'driven', 'and', 'computational', '##ly', '-', 'oriented', 'version', 'of', 'statistics', '.', 'both', 'the', 'term', 'data', 'science', 'and', 'the', 'broader', 'idea', 'it', 'convey', '##s', 'have', 'origins', 'in', 'statistics', 'and', 'are', 'a', 'reaction', 'to', 'a', 'narrower', 'view', 'of', 'data', 'analysis', '.', 'expanding', 'upon', 'the', 'views', 'of', 'a', 'number', 'of', 'stat', '##istic', '##ians', ',', 'this', 'paper', 'encourages', 'a', 'big', '-', 'tent', 'view', 'of', 'data', 'analysis', '.', 'we', 'examine', 'how', 'evolving', 'approaches', 'to', 'modern', 'data', 'analysis', 'relate', 'to', 'the', 'existing', 'discipline', 'of', 'statistics', '(', 'e', '.', 'g', '.', 'ex', '##pl', '##ora', '##tory', 

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [29]:
tokenizer = BertTokenizer.from_pretrained("bert-large-cased")
print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

['Data', 'science', 'is', 'the', 'business', 'of', 'learning', 'from', 'data', ',', 'which', 'is', 'traditionally', 'the', 'business', 'of', 'statistics', '.', 'Data', 'science', ',', 'however', ',', 'is', 'often', 'understood', 'as', 'a', 'broader', ',', 'task', '-', 'driven', 'and', 'computational', '##ly', '-', 'oriented', 'version', 'of', 'statistics', '.', 'Both', 'the', 'term', 'data', 'science', 'and', 'the', 'broader', 'idea', 'it', 'convey', '##s', 'have', 'origins', 'in', 'statistics', 'and', 'are', 'a', 'reaction', 'to', 'a', 'narrower', 'view', 'of', 'data', 'analysis', '.', 'Ex', '##pan', '##ding', 'upon', 'the', 'views', 'of', 'a', 'number', 'of', 's', '##tat', '##istic', '##ians', ',', 'this', 'paper', 'encourages', 'a', 'big', '-', 'tent', 'view', 'of', 'data', 'analysis', '.', 'We', 'examine', 'how', 'evolving', 'approaches', 'to', 'modern', 'data', 'analysis', 'relate', 'to', 'the', 'existing', 'discipline', 'of', 'statistics', '(', 'e', '.', 'g', '.', 'ex', '##p', '#

In [26]:
from transformers import GPT2Tokenizer, GPT2Model

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))



['Data', 'Ġscience', 'Ġis', 'Ġthe', 'Ġbusiness', 'Ġof', 'Ġlearning', 'Ġfrom', 'Ġdata', ',', 'Ġwhich', 'Ġis', 'Ġtraditionally', 'Ċ', 'the', 'Ġbusiness', 'Ġof', 'Ġstatistics', '.', 'ĠData', 'Ġscience', ',', 'Ġhowever', ',', 'Ġis', 'Ġoften', 'Ġunderstood', 'Ġas', 'Ġa', 'Ċ', 'bro', 'ader', ',', 'Ġtask', '-', 'driven', 'Ġand', 'Ġcomput', 'ationally', '-', 'oriented', 'Ġversion', 'Ġof', 'Ġstatistics', '.', 'ĠBoth', 'Ċ', 'the', 'Ġterm', 'Ġdata', 'Ġscience', 'Ġand', 'Ġthe', 'Ġbroader', 'Ġidea', 'Ġit', 'Ġconve', 'ys', 'Ġhave', 'Ġorigins', 'Ġin', 'Ċ', 'stat', 'istics', 'Ġand', 'Ġare', 'Ġa', 'Ġreaction', 'Ġto', 'Ġa', 'Ġnarrower', 'Ġview', 'Ġof', 'Ġdata', 'Ġanalysis', '.', 'ĠExp', 'anding', 'Ċ', 'upon', 'Ġthe', 'Ġviews', 'Ġof', 'Ġa', 'Ġnumber', 'Ġof', 'Ġstatistic', 'ians', ',', 'Ġthis', 'Ġpaper', 'Ġencourages', 'Ġa', 'Ġbig', '-', 't', 'ent', 'Ċ', 'view', 'Ġof', 'Ġdata', 'Ġanalysis', '.', 'ĠWe', 'Ġexamine', 'Ġhow', 'Ġevolving', 'Ġapproaches', 'Ġto', 'Ġmodern', 'Ġdata', 'Ċ', 'analysis', 'Ġrelate', '

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [27]:
from transformers import RobertaTokenizer, RobertaModel

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base")

print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

Error while downloading from https://cdn-lfs.hf.co/roberta-base/5bde1d28afb363d0103324efeb5afc8b2b397fe5e04beabb9b1ef355255ade81?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27model.safetensors%3B+filename%3D%22model.safetensors%22%3B&Expires=1735034824&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTczNTAzNDgyNH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yb2JlcnRhLWJhc2UvNWJkZTFkMjhhZmIzNjNkMDEwMzMyNGVmZWI1YWZjOGIyYjM5N2ZlNWUwNGJlYWJiOWIxZWYzNTUyNTVhZGU4MT9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19&Signature=ByZky%7ERGdjvaPce5ScKa60nLQg0UmiHq02OsqQThngUR%7ErFyxGl1zH-U9bByNlHrecq40Zlxf4cJITAhIpi5BQiWY0EC6BNPF5uE%7EuqBPpYe7YJ3xBLtJJn5mC9nWVRAFrizDQwZB2x5d3O1HQpPtPVMZxf74gOtjAmGY2Zk01x9GQzS2wEQbumiQwqSv8khuhYRw2QggENyQ7irrcQXFUmMlZ7lGXOZNDsl%7ERbUY1eknRle1%7E8C7Cmq2-gXq0zCYRsb2xP2KykJ18Nn64W6f4Hi5Q7c4vvWEVIuNz-XHC4dtP-FrLlL0cIglVL3EfGEogE2YBC-AFONgEkmsHyJkg__&Key-Pair-Id=K3RPWS32NSSJCE: HTTPSConnectionPool(host='cdn-lfs.hf.c

['Data', 'Ġscience', 'Ġis', 'Ġthe', 'Ġbusiness', 'Ġof', 'Ġlearning', 'Ġfrom', 'Ġdata', ',', 'Ġwhich', 'Ġis', 'Ġtraditionally', 'Ċ', 'the', 'Ġbusiness', 'Ġof', 'Ġstatistics', '.', 'ĠData', 'Ġscience', ',', 'Ġhowever', ',', 'Ġis', 'Ġoften', 'Ġunderstood', 'Ġas', 'Ġa', 'Ċ', 'bro', 'ader', ',', 'Ġtask', '-', 'driven', 'Ġand', 'Ġcomput', 'ationally', '-', 'oriented', 'Ġversion', 'Ġof', 'Ġstatistics', '.', 'ĠBoth', 'Ċ', 'the', 'Ġterm', 'Ġdata', 'Ġscience', 'Ġand', 'Ġthe', 'Ġbroader', 'Ġidea', 'Ġit', 'Ġconve', 'ys', 'Ġhave', 'Ġorigins', 'Ġin', 'Ċ', 'stat', 'istics', 'Ġand', 'Ġare', 'Ġa', 'Ġreaction', 'Ġto', 'Ġa', 'Ġnarrower', 'Ġview', 'Ġof', 'Ġdata', 'Ġanalysis', '.', 'ĠExp', 'anding', 'Ċ', 'upon', 'Ġthe', 'Ġviews', 'Ġof', 'Ġa', 'Ġnumber', 'Ġof', 'Ġstatistic', 'ians', ',', 'Ġthis', 'Ġpaper', 'Ġencourages', 'Ġa', 'Ġbig', '-', 't', 'ent', 'Ċ', 'view', 'Ġof', 'Ġdata', 'Ġanalysis', '.', 'ĠWe', 'Ġexamine', 'Ġhow', 'Ġevolving', 'Ġapproaches', 'Ġto', 'Ġmodern', 'Ġdata', 'Ċ', 'analysis', 'Ġrelate', '

In [28]:
from transformers import XLNetTokenizer, XLNetModel

tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")

print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))
# model = XLNetModel.from_pretrained("xlnet-base-cased")

['▁Data', '▁science', '▁is', '▁the', '▁business', '▁of', '▁learning', '▁from', '▁data', ',', '▁which', '▁is', '▁traditionally', '▁the', '▁business', '▁of', '▁statistics', '.', '▁Data', '▁science', ',', '▁however', ',', '▁is', '▁often', '▁understood', '▁as', '▁a', '▁broader', ',', '▁task', '-', 'driven', '▁and', '▁computational', 'ly', '-', 'oriented', '▁version', '▁of', '▁statistics', '.', '▁Both', '▁the', '▁term', '▁data', '▁science', '▁and', '▁the', '▁broader', '▁idea', '▁it', '▁convey', 's', '▁have', '▁origins', '▁in', '▁statistics', '▁and', '▁are', '▁a', '▁reaction', '▁to', '▁a', '▁narrow', 'er', '▁view', '▁of', '▁data', '▁analysis', '.', '▁Exp', 'and', 'ing', '▁upon', '▁the', '▁views', '▁of', '▁a', '▁number', '▁of', '▁', 'statistic', 'ians', ',', '▁this', '▁paper', '▁encourages', '▁a', '▁big', '-', 'tent', '▁view', '▁of', '▁data', '▁analysis', '.', '▁We', '▁examine', '▁how', '▁', 'evo', 'lving', '▁approaches', '▁to', '▁modern', '▁data', '▁analysis', '▁relate', '▁to', '▁the', '▁exi

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [30]:
from transformers import T5Tokenizer, T5EncoderModel

tokenizer = T5Tokenizer.from_pretrained("t5-small")

print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


['▁Data', '▁science', '▁is', '▁the', '▁business', '▁of', '▁learning', '▁from', '▁data', ',', '▁which', '▁is', '▁traditionally', '▁the', '▁business', '▁of', '▁statistics', '.', '▁Data', '▁science', ',', '▁however', ',', '▁is', '▁often', '▁understood', '▁as', '▁', 'a', '▁', 'broader', ',', '▁task', '-', 'driven', '▁and', '▁computational', 'ly', '-', 'oriented', '▁version', '▁of', '▁statistics', '.', '▁Both', '▁the', '▁term', '▁data', '▁science', '▁and', '▁the', '▁', 'broader', '▁idea', '▁it', '▁convey', 's', '▁have', '▁origin', 's', '▁in', '▁statistics', '▁and', '▁are', '▁', 'a', '▁reaction', '▁to', '▁', 'a', '▁narrow', 'er', '▁view', '▁of', '▁data', '▁analysis', '.', '▁Expand', 'ing', '▁upon', '▁the', '▁views', '▁of', '▁', 'a', '▁number', '▁of', '▁statistic', 'ians', ',', '▁this', '▁paper', '▁encourage', 's', '▁', 'a', '▁big', '-', 'tent', '▁view', '▁of', '▁data', '▁analysis', '.', '▁We', '▁examine', '▁how', '▁evolving', '▁approaches', '▁to', '▁modern', '▁data', '▁analysis', '▁relate', 

In [31]:
from transformers import AlbertTokenizer, AlbertModel

tokenizer = AlbertTokenizer.from_pretrained("albert-base-v2")

print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

['▁data', '▁science', '▁is', '▁the', '▁business', '▁of', '▁learning', '▁from', '▁data', ',', '▁which', '▁is', '▁traditionally', '▁the', '▁business', '▁of', '▁statistics', '.', '▁data', '▁science', ',', '▁however', ',', '▁is', '▁often', '▁understood', '▁as', '▁a', '▁broader', ',', '▁task', '-', 'driven', '▁and', '▁computational', 'ly', '-', 'oriented', '▁version', '▁of', '▁statistics', '.', '▁both', '▁the', '▁term', '▁data', '▁science', '▁and', '▁the', '▁broader', '▁idea', '▁it', '▁convey', 's', '▁have', '▁origins', '▁in', '▁statistics', '▁and', '▁are', '▁a', '▁reaction', '▁to', '▁a', '▁narrower', '▁view', '▁of', '▁data', '▁analysis', '.', '▁expanding', '▁upon', '▁the', '▁views', '▁of', '▁a', '▁number', '▁of', '▁statistic', 'ians', ',', '▁this', '▁paper', '▁encourages', '▁a', '▁big', '-', 'ten', 't', '▁view', '▁of', '▁data', '▁analysis', '.', '▁we', '▁examine', '▁how', '▁evolving', '▁approaches', '▁to', '▁modern', '▁data', '▁analysis', '▁relate', '▁to', '▁the', '▁existing', '▁discipline

In [32]:
from transformers import ElectraTokenizer, ElectraModel

tokenizer = ElectraTokenizer.from_pretrained("google/electra-small-discriminator")

print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

['data', 'science', 'is', 'the', 'business', 'of', 'learning', 'from', 'data', ',', 'which', 'is', 'traditionally', 'the', 'business', 'of', 'statistics', '.', 'data', 'science', ',', 'however', ',', 'is', 'often', 'understood', 'as', 'a', 'broader', ',', 'task', '-', 'driven', 'and', 'computational', '##ly', '-', 'oriented', 'version', 'of', 'statistics', '.', 'both', 'the', 'term', 'data', 'science', 'and', 'the', 'broader', 'idea', 'it', 'convey', '##s', 'have', 'origins', 'in', 'statistics', 'and', 'are', 'a', 'reaction', 'to', 'a', 'narrower', 'view', 'of', 'data', 'analysis', '.', 'expanding', 'upon', 'the', 'views', 'of', 'a', 'number', 'of', 'stat', '##istic', '##ians', ',', 'this', 'paper', 'encourages', 'a', 'big', '-', 'tent', 'view', 'of', 'data', 'analysis', '.', 'we', 'examine', 'how', 'evolving', 'approaches', 'to', 'modern', 'data', 'analysis', 'relate', 'to', 'the', 'existing', 'discipline', 'of', 'statistics', '(', 'e', '.', 'g', '.', 'ex', '##pl', '##ora', '##tory', 

In [33]:
from transformers import DistilBertTokenizer, DistilBertModel

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

['data', 'science', 'is', 'the', 'business', 'of', 'learning', 'from', 'data', ',', 'which', 'is', 'traditionally', 'the', 'business', 'of', 'statistics', '.', 'data', 'science', ',', 'however', ',', 'is', 'often', 'understood', 'as', 'a', 'broader', ',', 'task', '-', 'driven', 'and', 'computational', '##ly', '-', 'oriented', 'version', 'of', 'statistics', '.', 'both', 'the', 'term', 'data', 'science', 'and', 'the', 'broader', 'idea', 'it', 'convey', '##s', 'have', 'origins', 'in', 'statistics', 'and', 'are', 'a', 'reaction', 'to', 'a', 'narrower', 'view', 'of', 'data', 'analysis', '.', 'expanding', 'upon', 'the', 'views', 'of', 'a', 'number', 'of', 'stat', '##istic', '##ians', ',', 'this', 'paper', 'encourages', 'a', 'big', '-', 'tent', 'view', 'of', 'data', 'analysis', '.', 'we', 'examine', 'how', 'evolving', 'approaches', 'to', 'modern', 'data', 'analysis', 'relate', 'to', 'the', 'existing', 'discipline', 'of', 'statistics', '(', 'e', '.', 'g', '.', 'ex', '##pl', '##ora', '##tory', 

In [34]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("nghuyong/ernie-1.0-base-zh")

print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

['data', 'science', 'is', 'the', 'business', 'of', 'learning', 'from', 'data', ',', 'which', 'is', 't', '##ra', '##di', '##tional', '##ly', 'the', 'business', 'of', 'st', '##at', '##ist', '##ics', '.', 'data', 'science', ',', 'how', '##ever', ',', 'is', 'of', '##ten', 'under', '##st', '##ood', 'as', 'a', 'br', '##oa', '##der', ',', 'ta', '##sk', '[UNK]', 'drive', '##n', 'and', 'com', '##put', '##ation', '##all', '##y', '[UNK]', 'or', '##ien', '##ted', 'version', 'of', 'st', '##at', '##ist', '##ics', '.', 'both', 'the', 'te', '##rm', 'data', 'science', 'and', 'the', 'br', '##oa', '##der', 'idea', 'it', 'con', '##ve', '##ys', 'have', 'or', '##ig', '##ins', 'in', 'st', '##at', '##ist', '##ics', 'and', 'are', 'a', 're', '##act', '##ion', 'to', 'a', 'na', '##rr', '##ow', '##er', 'view', 'of', 'data', 'analysis', '.', 'ex', '##pan', '##ding', 'up', '##on', 'the', 'views', 'of', 'a', 'number', 'of', 'st', '##at', '##ist', '##ic', '##ian', '##s', ',', 'this', 'paper', 'en', '##co', '##ura', '#

In [45]:
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
# model = BertModel.from_pretrained("your-fine-tuned-spert-model")

print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

['data', 'science', 'is', 'the', 'business', 'of', 'learning', 'from', 'data', ',', 'which', 'is', 'traditionally', 'the', 'business', 'of', 'statistics', '.', 'data', 'science', ',', 'however', ',', 'is', 'often', 'understood', 'as', 'a', 'broader', ',', 'task', '-', 'driven', 'and', 'computational', '##ly', '-', 'oriented', 'version', 'of', 'statistics', '.', 'both', 'the', 'term', 'data', 'science', 'and', 'the', 'broader', 'idea', 'it', 'convey', '##s', 'have', 'origins', 'in', 'statistics', 'and', 'are', 'a', 'reaction', 'to', 'a', 'narrower', 'view', 'of', 'data', 'analysis', '.', 'expanding', 'upon', 'the', 'views', 'of', 'a', 'number', 'of', 'stat', '##istic', '##ians', ',', 'this', 'paper', 'encourages', 'a', 'big', '-', 'tent', 'view', 'of', 'data', 'analysis', '.', 'we', 'examine', 'how', 'evolving', 'approaches', 'to', 'modern', 'data', 'analysis', 'relate', 'to', 'the', 'existing', 'discipline', 'of', 'statistics', '(', 'e', '.', 'g', '.', 'ex', '##pl', '##ora', '##tory', 

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [71]:
text = df["Abstract"].values[0]

In [72]:
print(tokenizer.tokenize(text))
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

['data', 'science', 'is', 'the', 'business', 'of', 'learning', 'from', 'data', ',', 'which', 'is', 'traditionally', 'the', 'business', 'of', 'statistics', '.', 'data', 'science', ',', 'however', ',', 'is', 'often', 'understood', 'as', 'a', 'broader', ',', 'task', '-', 'driven', 'and', 'computational', '##ly', '-', 'oriented', 'version', 'of', 'statistics', '.', 'both', 'the', 'term', 'data', 'science', 'and', 'the', 'broader', 'idea', 'it', 'convey', '##s', 'have', 'origins', 'in', 'statistics', 'and', 'are', 'a', 'reaction', 'to', 'a', 'narrower', 'view', 'of', 'data', 'analysis', '.', 'expanding', 'upon', 'the', 'views', 'of', 'a', 'number', 'of', 'stat', '##istic', '##ians', ',', 'this', 'paper', 'encourages', 'a', 'big', '-', 'tent', 'view', 'of', 'data', 'analysis', '.', 'we', 'examine', 'how', 'evolving', 'approaches', 'to', 'modern', 'data', 'analysis', 'relate', 'to', 'the', 'existing', 'discipline', 'of', 'statistics', '(', 'e', '.', 'g', '.', 'ex', '##pl', '##ora', '##tory', 

In [ ]:
from transformers import BertTokenizer
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Sample DataFrame (replace this with your actual DataFrame)
data = {
    "Abstract": [
        "This is the first abstract.",
        "Another example of an abstract for classification.",
        "This is the third abstract used for training."
    ],
    "Label": [0, 1, 0]  # Binary labels
}
df = pd.DataFrame(data)

# Parameters
max_length = 128  # Fixed sequence length for BERT input

# Tokenize all abstracts and convert them into fixed-length sequences
def tokenize_and_encode(df, tokenizer, max_length):
    input_ids = []
    attention_masks = []

    for abstract in df["Abstract"]:
        # Tokenize and encode the text
        encoded = tokenizer.encode_plus(
            abstract,
            add_special_tokens=True,  # Adds [CLS] and [SEP] tokens
            max_length=max_length,  # Pad/Truncate to this length
            padding='max_length',  # Pad to max_length
            truncation=True,  # Truncate if too long
            return_attention_mask=True,  # Include attention mask
            return_tensors="pt"  # Return as PyTorch tensors
        )
        input_ids.append(encoded["input_ids"])
        attention_masks.append(encoded["attention_mask"])

    return torch.cat(input_ids, dim=0), torch.cat(attention_masks, dim=0)

# Encode the dataset
input_ids, attention_masks = tokenize_and_encode(df, tokenizer, max_length)

# Convert labels to tensor
labels = torch.tensor(df["Label"].values)

# Display encoded inputs
print("Input IDs:\n", input_ids)
print("Attention Masks:\n", attention_masks)
print("Labels:\n", labels)

# Dataset and DataLoader for training
class CustomDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_masks[idx],
            "label": self.labels[idx]
        }

# Create a Dataset object
dataset = CustomDataset(input_ids, attention_masks, labels)

# Create a DataLoader
batch_size = 8
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Display a batch
for batch in dataloader:
    print(batch)
    break
